# 03. Limpieza conservadora

Se retira únicamente material que pertenece a la plantilla de publicación y no al mensaje: `Confesión #1835`, `Susurro #...`, el par inicial `Carrera/Semestre` asociado a esa plantilla y un rótulo inicial repetido. Después se ordenan espacios.

No se eliminan emojis, negaciones, signos, hashtags, jerga ni palabras comunes porque pueden cambiar la polaridad. El texto original siempre se conserva en `texto_real`.


La reproducción lee las exportaciones guardadas y escribe en `salidas/<ejecución>/TABLAS`.

Si abres este cuaderno directamente en Colab, ejecuta su primera celda y autoriza Google Drive cuando se solicite. Para recorrer toda la secuencia de una vez, usa `INICIAR_EN_COLAB.ipynb`.

## A. Cargar los datos

- **Qué hace:** lee las tres columnas producidas por el cuaderno 01.
- **Para qué:** iniciar la limpieza desde el texto real.
- **Por qué:** así se puede comparar cada cambio con el original y recuperar el dato si una regla resulta incorrecta.


In [ ]:
# Inicio al abrir este cuaderno directamente en Google Colab.
from pathlib import Path

if not Path("corrida.json").is_file():
    import os
    import subprocess
    import sys

    try:
        from google.colab import drive
    except ImportError as error:
        raise FileNotFoundError("En Jupyter, abre la carpeta que contiene corrida.json.") from error

    proyecto_drive = Path("/content/drive/MyDrive/Tesis_RoBERTuito_M0_M1_M2")
    if not (proyecto_drive / "corrida.json").is_file():
        drive.mount("/content/drive", timeout_ms=180000)
    if not (proyecto_drive / "corrida.json").is_file():
        raise FileNotFoundError("Busca Tesis_RoBERTuito_M0_M1_M2 directamente en Mi unidad.")

    os.chdir(proyecto_drive)
    sys.path.insert(0, str(proyecto_drive))
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r",
         str(proyecto_drive / "requirements-cuadernos.txt")],
        check=True,
    )
    print("Carpeta de la tesis conectada:", proyecto_drive)

from rutas import carpeta_salidas

from pathlib import Path

RAIZ = Path.cwd()
if not (RAIZ / "corrida.json").is_file():
    raise FileNotFoundError("Abre Jupyter en la carpeta que contiene corrida.json.")
print("Carpeta de trabajo:", RAIZ)

import json
import re
import pandas as pd
from IPython.display import display

ENTRADA = carpeta_salidas(RAIZ) / "TABLAS/01_integracion/datos_integrados_v4.xlsx"
datos = pd.read_excel(ENTRADA, sheet_name="datos", keep_default_na=False)
if list(datos.columns) != ["id", "fuente", "texto_real"]:
    raise ValueError("Ejecuta primero el cuaderno 01 actualizado.")

trabajo = datos.copy()
trabajo["texto_limpio"] = trabajo["texto_real"].astype(str)
print("Entrada: primeras 10 filas")
display(trabajo.head(10))


## B. Quitar la cabecera numerada

- **Qué hace:** elimina `Confesión` o `Susurro` seguido de un número solamente cuando aparece al inicio.
- **Para qué:** evitar que el nombre y número de la plantilla influyan en la clasificación.
- **Por qué:** esa cabecera organiza la publicación, pero no expresa la opinión del autor.


In [ ]:
PATRON_CABECERA = re.compile(
    r"^\s*(?:susurro|confesi[oó]n)\s*:?[#\s]*\d+\b(?:\s*[-–]\s*continuaci[oó]n)?\s*:?\s*",
    re.IGNORECASE,
)
tenia_cabecera = trabajo["texto_limpio"].str.match(PATRON_CABECERA)
trabajo["texto_limpio"] = trabajo["texto_limpio"].str.replace(PATRON_CABECERA, "", regex=True)
n_cabeceras = int(tenia_cabecera.sum())
print("Cabeceras retiradas:", n_cabeceras)
display(trabajo[["id", "fuente", "texto_real", "texto_limpio"]].head(10))


## C. Quitar `Carrera` y `Semestre` de la plantilla

- **Qué hace:** retira esos dos campos iniciales solo en textos de Telegram que tenían cabecera numerada.
- **Para qué:** eliminar metadatos copiados dentro del mensaje.
- **Por qué:** restringir la regla evita borrar frases normales como “mi carrera es Informática”.


In [ ]:
PATRON_METADATOS = re.compile(
    r"^\s*Carrera[ \t]*:[^\r\n]{0,100}?(?:\r?\n[ \t]*|(?=Semestre[ \t]*:))"
    r"Semestre[ \t]*:[^\r\n]{0,100}(?:\r?\n|$)",
    re.IGNORECASE,
)
aplica_metadatos = trabajo["fuente"].eq("telegram") & tenia_cabecera
tenia_metadatos = aplica_metadatos & trabajo["texto_limpio"].str.match(PATRON_METADATOS)
trabajo.loc[aplica_metadatos, "texto_limpio"] = (
    trabajo.loc[aplica_metadatos, "texto_limpio"].str.replace(PATRON_METADATOS, "", regex=True)
)
n_metadatos = int(tenia_metadatos.sum())
print("Bloques Carrera/Semestre retirados:", n_metadatos)
display(trabajo[["id", "fuente", "texto_real", "texto_limpio"]].head(10))


## D. Quitar un rótulo inicial repetido

- **Qué hace:** elimina un segundo `Susurro:` o `Confesión:` al inicio, únicamente en publicaciones que tenían cabecera.
- **Para qué:** dejar el contenido del mensaje sin el nombre duplicado de la sección.
- **Por qué:** la condición conserva usos normales de esas palabras dentro del texto.


In [ ]:
def quitar_rotulo_inicial(texto):
    texto = texto.lstrip()
    rotulos = ["susurro:", "confesión:", "confesion:"]

    for rotulo in rotulos:
        if texto.casefold().startswith(rotulo):
            return texto[len(rotulo):].lstrip()

    return texto

filas_con_cabecera = trabajo.index[tenia_cabecera]
antes = trabajo.loc[filas_con_cabecera, "texto_limpio"].copy()
despues = antes.map(quitar_rotulo_inicial)
trabajo.loc[filas_con_cabecera, "texto_limpio"] = despues

n_rotulos = int((antes != despues).sum())
print("Rótulos repetidos retirados:", n_rotulos)
display(trabajo[["id", "fuente", "texto_real", "texto_limpio"]].head(10))


## E. Ordenar espacios

- **Qué hace:** convierte saltos y grupos de espacios en un solo espacio, y quita espacios de los extremos.
- **Para qué:** producir una representación uniforme para el procesamiento posterior.
- **Por qué:** las diferencias accidentales de formato no deben crear textos distintos.


In [ ]:
trabajo["texto_limpio"] = trabajo["texto_limpio"].str.replace(r"\s+", " ", regex=True).str.strip()

if len(trabajo) != len(datos) or trabajo["id"].tolist() != datos["id"].tolist():
    raise ValueError("La limpieza no debe perder ni reordenar filas.")
if PATRON_CABECERA.match("Mi confesión fue triste"):
    raise ValueError("La regla de cabecera es demasiado amplia.")

print("Resultado de la limpieza: primeras 10 filas")
display(trabajo[["id", "fuente", "texto_real", "texto_limpio"]].head(10))


## F. Guardar la salida mínima

- **Qué hace:** exporta cuatro columnas y guarda los conteos de limpieza en un JSON separado.
- **Para qué:** entregar al cuaderno 04 tanto el original como la versión limpia.
- **Por qué:** las banderas usadas para contar cambios sirven para auditoría, no como variables del modelo.


In [ ]:
from rutas import carpeta_salidas

salida = trabajo[["id", "fuente", "texto_real", "texto_limpio"]].copy()
CARPETA = carpeta_salidas(RAIZ) / "TABLAS/03_limpieza"
CARPETA.mkdir(parents=True, exist_ok=True)
ARCHIVO = CARPETA / "datos_limpios_v4.xlsx"
RESUMEN = CARPETA / "resumen_limpieza_v4.json"

salida.to_excel(ARCHIVO, sheet_name="datos", index=False)
resumen = {
    "version": "v4",
    "filas_entrada": int(len(datos)),
    "filas_salida": int(len(salida)),
    "cabeceras_retiradas": n_cabeceras,
    "bloques_carrera_semestre_retirados": n_metadatos,
    "rotulos_repetidos_retirados": n_rotulos,
    "textos_vacios_despues": int(salida["texto_limpio"].eq("").sum()),
    "columnas_salida": list(salida.columns),
}
RESUMEN.write_text(json.dumps(resumen, ensure_ascii=False, indent=2), encoding="utf-8")
print("Guardado:", ARCHIVO)
print("Columnas finales:", list(salida.columns))
display(salida.head(10))
